# Method 1: MAE (self-supervised, ViT)
Fine-tune a pretrained Masked Autoencoder on clean AFHQ (hide ~75% of patches, MSE on hidden patches). At test time every patch overlapping the hole is treated as hidden. Deterministic, fast.

**Code lives in** `src/methods/mae.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas matplotlib
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !{sys.executable} scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Model + fine-tune

In [ ]:
from src.methods.mae import MAEInpainter
mae = MAEInpainter(device=dev)
mae.train(train_loader, epochs=10)   # saves checkpoint to checkpoints/mae/

## 3. Evaluate (fixed test masks)

In [ ]:
from src.eval import evaluate, save_result
for mask in ['square', 'one_sided', 'centre']:
    res = evaluate(mae, mask, n_images=200, device=dev, save_dir=f'outputs/samples/{mae.name}_{mask}')
    save_result(mae.name, mask, res)
    print(mask, res)